# EV Charging Station Placement: A Capacitated Facility Location Approach
**Author:** Omer Toledo | **Date:** May 2026

This notebook implements the full pipeline for optimizing public EV charging station placement in Mountain View, California using a Capacitated Facility Location Problem (CFLP) formulation.

**Pipeline order:**
1. Demand points — Census block groups + ACS vehicle ownership + tenure data (loaded from GitHub)
2. Home-Charging Vulnerability Index (HCVI) — loaded from GitHub
3. Candidate sites — OSM parking lots + feasibility tiers (loaded from GitHub)
4. Distance matrix — haversine, 3 km proximity filter
5. MILP solver — Scenario A (uniform 10% adoption), proven optimal
6. MILP solver — Scenario B (tenure-weighted demand), feasible solution
7. Heuristic — Coverage-first greedy + simulated annealing
8. Existing network baseline + incremental expansion (33-station AFDC network)
9. Equity analysis — HCVI-stratified metrics + equity-constrained MILP
10. Road-network distance validation — haversine vs. OSMnx shortest paths
11. Sensitivity grid — K, Q, r, adoption rate
12. Stochastic demand analysis — 6 scenarios, regret, CVaR_0.90, site stability
13. Queueing / wait-time model — M/M/m Erlang C, port sensitivity, Scenario B stress test
14. Minimax-regret analysis — attempted; exceeds solver budget at this instance scale (see limitations)
15. Interactive solution maps (Folium)
16. File verification + ZIP download


In [2]:
# ================================
# CELL 1: Install dependencies and load data from GitHub
# ================================
!pip install -q osmnx ortools folium

import pandas as pd
import numpy as np

BASE_URL = "https://raw.githubusercontent.com/otoledo1/ev-cflp-mountain-view/main/data/processed/"

result   = pd.read_csv(BASE_URL + "Mountain_View_demand_points.csv")
sites    = pd.read_csv(BASE_URL + "Mountain_View_candidate_sites.csv")
existing = pd.read_csv(BASE_URL + "Mountain_View_existing_stations.csv")
hcvi_df  = pd.read_csv(BASE_URL + "Mountain_View_HCVI.csv")

print(f"Loaded {len(result)} block groups")
print(f"Loaded {len(sites)} candidate sites")
print(f"Columns: {result.columns.tolist()}")

# Reconstruct demand vectors
EV_ADOPTION_RATE = 0.10
OWNER_RATE       = 0.05
RENTER_RATE      = 0.30

result["owner_vehicles"]  = result["vehicles"] * (1 - result["renter_share"])
result["renter_vehicles"] = result["vehicles"] * result["renter_share"]
result["demand_A"] = result["vehicles"] * EV_ADOPTION_RATE
result["demand_B"] = (
    result["owner_vehicles"]  * OWNER_RATE +
    result["renter_vehicles"] * RENTER_RATE
)
result["demand"] = result["demand_A"]

OLD_COUNT  = 240
OLD_DEMAND = 15663.6

print(f"\n── Geography Audit ─────────────────────────────────────────────")
print(f"  Block groups: {OLD_COUNT} (bbox) → {len(result)} (city boundary)")
print(f"  Total demand (Scenario A): {OLD_DEMAND} → {result['demand_A'].sum():.1f}")
print(f"────────────────────────────────────────────────────────────────")
print(f"\n── Demand Scenario Comparison ───────────────────────────────────")
print(f"  {'Metric':<40} {'Scenario A':>12} {'Scenario B':>12}")
print(f"  {'-'*64}")
print(f"  {'Total demand (units)':<40} {result['demand_A'].sum():>12.1f} {result['demand_B'].sum():>12.1f}")
print(f"  {'Mean per block group':<40} {result['demand_A'].mean():>12.1f} {result['demand_B'].mean():>12.1f}")
print(f"  {'Max per block group':<40} {result['demand_A'].max():>12.1f} {result['demand_B'].max():>12.1f}")
print(f"  {'Mean renter share':<40} {result['renter_share'].mean():>12.3f} {'(weighted)':>12}")
print(f"────────────────────────────────────────────────────────────────")
print(f"\nSample (first 5 rows):")
print(result[["block_group","lat","lon","demand_A","demand_B","renter_share"]].head().to_string())

Loaded 79 block groups
Loaded 792 candidate sites
Columns: ['block_group', 'lat', 'lon', 'population', 'vehicles', 'renter_share', 'demand_A', 'demand_B', 'demand']

── Geography Audit ─────────────────────────────────────────────
  Block groups: 240 (bbox) → 79 (city boundary)
  Total demand (Scenario A): 15663.6 → 5339.4
────────────────────────────────────────────────────────────────

── Demand Scenario Comparison ───────────────────────────────────
  Metric                                     Scenario A   Scenario B
  ----------------------------------------------------------------
  Total demand (units)                           5339.4      10518.1
  Mean per block group                             67.6        133.1
  Max per block group                             153.2        415.5
  Mean renter share                               0.522   (weighted)
────────────────────────────────────────────────────────────────

Sample (first 5 rows):
   block_group        lat         lon  dem

In [3]:
# ================================
# CELL 2: Home-Charging Vulnerability Index (HCVI)
# Loads from pre-computed CSV; no Census API required
# ================================
import numpy as np
import pandas as pd

# Merge HCVI onto result
result_hcvi = result.merge(hcvi_df, on="block_group", how="left")

# Define high-vulnerability set H (top quartile by HCVI_norm)
hcvi_threshold = result_hcvi["HCVI_norm"].quantile(0.75)
result_hcvi["high_vulnerability"] = result_hcvi["HCVI_norm"] >= hcvi_threshold

H_idx = result_hcvi.index[result_hcvi["high_vulnerability"]].tolist()
H_set = set(H_idx)

print(f"── Home-Charging Vulnerability Index ───────────────────────────")
print(f"  Block groups total:           {len(result_hcvi)}")
print(f"  High-vulnerability set |H|:   {len(H_idx)} (top quartile)")
print(f"  HCVI threshold (75th pct):    {hcvi_threshold:.3f}")

components = ["c1_renter", "c2_income", "c3_multifamily", "c4_zero_vehicle"]
existing_components = [c for c in components if c in result_hcvi.columns]
if existing_components:
    print(f"\n  Component means:")
    for c in existing_components:
        print(f"    {c}: {result_hcvi[c].mean():.3f}")

print(f"\n  HCVI stats:")
print(f"    Mean:   {result_hcvi['HCVI_norm'].mean():.3f}")
print(f"    Median: {result_hcvi['HCVI_norm'].median():.3f}")
print(f"    Min:    {result_hcvi['HCVI_norm'].min():.3f}")
print(f"    Max:    {result_hcvi['HCVI_norm'].max():.3f}")
print(f"\nH_idx defined: {len(H_idx)} block groups in high-vulnerability set")

── Home-Charging Vulnerability Index ───────────────────────────
  Block groups total:           79
  High-vulnerability set |H|:   20 (top quartile)
  HCVI threshold (75th pct):    0.767

  Component means:
    c1_renter: 0.522
    c2_income: 0.344
    c3_multifamily: 0.609
    c4_zero_vehicle: 0.536

  HCVI stats:
    Mean:   0.525
    Median: 0.577
    Min:    0.000
    Max:    1.000

H_idx defined: 20 block groups in high-vulnerability set


In [4]:
# ================================
# CELL 3: Candidate sites — load from GitHub
# (sites already loaded in Cell 1; this cell adds feasibility tiers)
# ================================
import pandas as pd

site_tiers = pd.read_csv(BASE_URL + "Mountain_View_site_tiers.csv")
sites = sites.merge(site_tiers[["site_id", "tier"]], on="site_id", how="left")

tier_labels = {
    1: "Municipal/government",
    2: "Retail/commercial",
    3: "Surface lot / unknown operator",
    4: "Underground/structured parking",
    5: "Private/restricted access"
}

print(f"── Candidate Site Feasibility Tiers ─────────────────────────────")
print(f"  {'Tier':<6} {'Label':<40} {'Count':>6} {'%':>6}")
print(f"  {'-'*58}")
tier_counts = sites["tier"].value_counts().sort_index()
for tier, count in tier_counts.items():
    pct = count / len(sites) * 100
    label = tier_labels.get(int(tier), "Unknown")
    print(f"  {int(tier):<6} {label:<40} {count:>6} {pct:>5.1f}%")
print(f"  {'-'*58}")
print(f"  {'Total':<46} {len(sites):>6}")
print(f"────────────────────────────────────────────────────────────────")
print(f"\nTotal candidate sites |J| = {len(sites)}")

── Candidate Site Feasibility Tiers ─────────────────────────────
  Tier   Label                                     Count      %
  ----------------------------------------------------------
  1      Municipal/government                          4   0.5%
  3      Surface lot / unknown operator              567  71.6%
  4      Underground/structured parking               18   2.3%
  5      Private/restricted access                   203  25.6%
  ----------------------------------------------------------
  Total                                             792
────────────────────────────────────────────────────────────────

Total candidate sites |J| = 792


In [5]:
# ================================
# CELL 4: Distance matrix
# ================================
import numpy as np
import pandas as pd

I = result[["block_group", "lat", "lon"]].copy()
J = sites[["site_id", "lat", "lon"]].copy()

def haversine_matrix(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2[:, None] - lat1[None, :]
    dlon = lon2[:, None] - lon1[None, :]
    a = (np.sin(dlat/2)**2 +
         np.cos(lat1[None, :]) * np.cos(lat2[:, None]) * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(a))

D = haversine_matrix(
    I["lat"].values, I["lon"].values,
    J["lat"].values, J["lon"].values
).T  # shape: (|I|, |J|)

print(f"Distance matrix shape: {D.shape}")
print(f"Min: {D.min():.4f} km  Max: {D.max():.4f} km  Mean: {D.mean():.4f} km")

MAX_DIST_KM = 3.0
valid_pairs    = [(i, j) for i in range(len(I))
                  for j in range(len(J)) if D[i, j] <= MAX_DIST_KM]
reachable_mask = [(D[i, :] <= MAX_DIST_KM).any() for i in range(len(I))]
n_reachable    = sum(reachable_mask)
n_unreachable  = len(I) - n_reachable

print(f"\nValid pairs within {MAX_DIST_KM} km: {len(valid_pairs)}")
print(f"Reduction from full matrix: {(1 - len(valid_pairs)/(len(I)*len(J)))*100:.1f}%")

print(f"\n── Geography Audit (Complete) ──────────────────────────────────────")
print(f"  {'Metric':<45} {'Old (bbox)':>12} {'New (boundary)':>15}")
print(f"  {'-'*72}")
print(f"  {'Block groups |I|':<45} {240:>12} {len(I):>15}")
print(f"  {'Total EV demand (Scenario A)':<45} {15663.6:>12.1f} {result['demand_A'].sum():>15.1f}")
print(f"  {'Total EV demand (Scenario B)':<45} {'—':>12} {result['demand_B'].sum():>15.1f}")
print(f"  {'Valid pairs within 3 km':<45} {37300:>12} {len(valid_pairs):>15}")
print(f"  {'Reachable block groups |I*|':<45} {143:>12} {n_reachable:>15}")
print(f"  {'Unreachable block groups':<45} {97:>12} {n_unreachable:>15}")
print(f"  {'-'*72}")

Distance matrix shape: (79, 792)
Min: 0.0221 km  Max: 9.5984 km  Mean: 3.2058 km

Valid pairs within 3.0 km: 30997
Reduction from full matrix: 50.5%

── Geography Audit (Complete) ──────────────────────────────────────
  Metric                                          Old (bbox)  New (boundary)
  ------------------------------------------------------------------------
  Block groups |I|                                       240              79
  Total EV demand (Scenario A)                       15663.6          5339.4
  Total EV demand (Scenario B)                             —         10518.1
  Valid pairs within 3 km                              37300           30997
  Reachable block groups |I*|                            143              79
  Unreachable block groups                                97               0
  ------------------------------------------------------------------------


In [6]:
# ================================
# CELL 5: MILP — Scenario A (primary method)
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import time

K          = 20
CAP        = 500
FIXED_COST = 50000

num_I = len(result)
num_J = len(sites)
I_idx = range(num_I)
J_idx = range(num_J)

d = result["demand_A"].values

neighbors_of_i = {i: [j for j in J_idx if D[i, j] <= MAX_DIST_KM] for i in I_idx}
neighbors_of_j = {j: [i for i in I_idx if D[i, j] <= MAX_DIST_KM] for j in J_idx}
valid_pairs    = [(i, j) for i in I_idx for j in neighbors_of_i[i]]
reachable      = [i for i in I_idx if neighbors_of_i[i]]
unreachable    = [i for i in I_idx if not neighbors_of_i[i]]

print(f"Valid pairs: {len(valid_pairs)}")
print(f"Reachable: {len(reachable)} / {num_I}")
print(f"Unreachable: {len(unreachable)}")

solver = pywraplp.Solver.CreateSolver("SCIP")
solver.SetTimeLimit(300_000)

x = [solver.BoolVar(f"x_{j}") for j in J_idx]
y = {(i, j): solver.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs}
u = [solver.BoolVar(f"u_{i}") for i in I_idx]

M_penalty = float(MAX_DIST_KM * max(d))
objective  = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(y[i, j], float(d[i]) * float(D[i, j]))
for i in I_idx:
    objective.SetCoefficient(u[i], M_penalty * float(d[i]))
objective.SetMinimization()

for i in I_idx:
    if neighbors_of_i[i]:
        solver.Add(sum(y[i, j] for j in neighbors_of_i[i]) + u[i] == 1)
    else:
        solver.Add(u[i] == 1)

for (i, j) in valid_pairs:
    solver.Add(y[i, j] <= x[j])

solver.Add(sum(x[j] for j in J_idx) == K)

for j in J_idx:
    if neighbors_of_j[j]:
        solver.Add(sum(d[i] * y[i, j] for i in neighbors_of_j[j]) <= CAP)

print(f"Variables:   {solver.NumVariables()}")
print(f"Constraints: {solver.NumConstraints()}")
print("Solving (Scenario A)...")

t0     = time.time()
status = solver.Solve()
solve_time = time.time() - t0

print(f"\n── Solver Audit (Scenario A) ────────────────────────────────────")
print(f"  Solver:              OR-Tools SCIP v9.x")
print(f"  Variables:           {solver.NumVariables()} (all binary)")
print(f"  Constraints:         {solver.NumConstraints()}")
print(f"  Status:              {'Optimal' if status == pywraplp.Solver.OPTIMAL else 'Feasible'}")
print(f"  Objective value:     {solver.Objective().Value():,.4f} EV-vehicle·km")
print(f"  Best bound:          {solver.Objective().BestBound():,.4f} EV-vehicle·km")
if solver.Objective().BestBound() > 0:
    mip_gap = abs(solver.Objective().Value() - solver.Objective().BestBound()) / abs(solver.Objective().Value())
    print(f"  MIP gap:             {mip_gap*100:.4f}%")
else:
    print(f"  MIP gap:             0.0000% (proven optimal)")
print(f"  Solve time:          {solve_time:.2f}s")
print(f"  Time limit:          300s")
print(f"  Hardware:            Google Colab (Intel Xeon, single thread)")
print(f"────────────────────────────────────────────────────────────────")

if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
    open_sites      = [j for j in J_idx if x[j].solution_value() > 0.5]
    unserved_idx    = [i for i in I_idx if u[i].solution_value() > 0.5]
    unserved_demand = sum(d[i] for i in unserved_idx)
    total_demand    = sum(d)
    served_demand   = total_demand - unserved_demand

    travel_cost = sum(
        d[i] * D[i, j] * y[i, j].solution_value()
        for (i, j) in valid_pairs
    )

    print(f"Stations built:   {len(open_sites)} / {K}")
    print(f"Unserved:         {len(unserved_idx)} / {num_I}")
    print(f"Served demand:    {served_demand:.1f} / {total_demand:.1f}")
    print(f"Fixed cost:       ${len(open_sites) * FIXED_COST:,.0f}")
    print(f"Travel cost:      {travel_cost:,.2f} EV-vehicle·km")

    rows = []
    for j in open_sites:
        assigned   = [i for i in neighbors_of_j[j] if y[i, j].solution_value() > 0.5]
        stn_demand = sum(d[i] for i in assigned)
        avg_dist   = (sum(D[i, j] for i in assigned) / len(assigned)
                      if assigned else 0.0)
        rows.append({
            "site_id":     sites["site_id"].iloc[j],
            "lat":         sites["lat"].iloc[j],
            "lon":         sites["lon"].iloc[j],
            "n_assigned":  len(assigned),
            "stn_demand":  round(stn_demand, 1),
            "avg_dist_km": round(avg_dist, 3),
            "load_pct":    round(stn_demand / CAP * 100, 1)
        })

    solution_df = pd.DataFrame(rows)

    # Store open site indices for downstream cells
    xA_open_indices = frozenset(open_sites)
    xA_open_ids     = set(solution_df["site_id"])

    avg_assign_dist = travel_cost / served_demand if served_demand > 0 else 0
    avg_load        = served_demand / len(open_sites) if open_sites else 0

    print(f"\nAvg. assignment distance: {avg_assign_dist:.3f} km")
    print(f"Avg. station load:        {avg_load:.1f} EV units ({avg_load/CAP*100:.1f}% of cap)")
    print(f"Solve time:               {solve_time:.1f}s")
    print("\nSelected stations:")
    print(solution_df.to_string(index=False))

    solution_df.to_csv("Mountain_View_CFLP_solution.csv", index=False)
    print("\nSaved to Mountain_View_CFLP_solution.csv")

Valid pairs: 30997
Reachable: 79 / 79
Unreachable: 0
Variables:   31868
Constraints: 31869
Solving (Scenario A)...

── Solver Audit (Scenario A) ────────────────────────────────────
  Solver:              OR-Tools SCIP v9.x
  Variables:           31868 (all binary)
  Constraints:         31869
  Status:              Optimal
  Objective value:     2,513.2618 EV-vehicle·km
  Best bound:          2,513.2618 EV-vehicle·km
  MIP gap:             0.0000%
  Solve time:          4.64s
  Time limit:          300s
  Hardware:            Google Colab (Intel Xeon, single thread)
────────────────────────────────────────────────────────────────
Stations built:   20 / 20
Unserved:         0 / 79
Served demand:    5339.4 / 5339.4
Fixed cost:       $1,000,000
Travel cost:      2,513.26 EV-vehicle·km

Avg. assignment distance: 0.471 km
Avg. station load:        267.0 EV units (53.4% of cap)
Solve time:               4.6s

Selected stations:
site_id       lat         lon  n_assigned  stn_demand  avg_dist

In [7]:
# ================================
# CELL 6: MILP — Scenario B (tenure-weighted demand)
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import time

d_B = result["demand_B"].values

print(f"Scenario A total demand: {result['demand_A'].sum():.1f} units")
print(f"Scenario B total demand: {result['demand_B'].sum():.1f} units")
print(f"Mean renter share: {result['renter_share'].mean():.3f}")

valid_pairs_B = [(i, j) for i in I_idx for j in neighbors_of_i[i]]

solver_B = pywraplp.Solver.CreateSolver("SCIP")
solver_B.SetTimeLimit(300_000)

x_B = [solver_B.BoolVar(f"x_{j}") for j in J_idx]
y_B = {(i, j): solver_B.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs_B}
u_B = [solver_B.BoolVar(f"u_{i}") for i in I_idx]

M_B   = float(MAX_DIST_KM * max(d_B))
obj_B = solver_B.Objective()
for (i, j) in valid_pairs_B:
    obj_B.SetCoefficient(y_B[i, j], float(d_B[i]) * float(D[i, j]))
for i in I_idx:
    obj_B.SetCoefficient(u_B[i], M_B * float(d_B[i]))
obj_B.SetMinimization()

for i in I_idx:
    if neighbors_of_i[i]:
        solver_B.Add(sum(y_B[i, j] for j in neighbors_of_i[i]) + u_B[i] == 1)
    else:
        solver_B.Add(u_B[i] == 1)

for (i, j) in valid_pairs_B:
    solver_B.Add(y_B[i, j] <= x_B[j])

solver_B.Add(sum(x_B[j] for j in J_idx) == K)

for j in J_idx:
    if neighbors_of_j[j]:
        solver_B.Add(sum(d_B[i] * y_B[i, j] for i in neighbors_of_j[j]) <= CAP)

print(f"\nVariables:   {solver_B.NumVariables()}")
print(f"Constraints: {solver_B.NumConstraints()}")
print("Solving (Scenario B)...")

t0_B     = time.time()
solverParams = pywraplp.MPSolverParameters()
solverParams.SetDoubleParam(pywraplp.MPSolverParameters.RELATIVE_MIP_GAP, 0.01)
status_B = solver_B.Solve(solverParams)
solve_B  = time.time() - t0_B

print(f"\n── Solver Audit (Scenario B) ────────────────────────────────────")
print(f"  Variables:           {solver_B.NumVariables()} (all binary)")
print(f"  Constraints:         {solver_B.NumConstraints()}")
print(f"  Status:              {'Optimal' if status_B == pywraplp.Solver.OPTIMAL else 'Feasible (time limit)'}")
print(f"  Objective value:     {solver_B.Objective().Value():,.4f} EV-vehicle·km")
print(f"  Best bound:          {solver_B.Objective().BestBound():,.4f} EV-vehicle·km")
if solver_B.Objective().BestBound() > 0 and solver_B.Objective().Value() > 0:
    mip_gap_B = abs(solver_B.Objective().Value() - solver_B.Objective().BestBound()) / abs(solver_B.Objective().Value())
    print(f"  MIP gap:             {mip_gap_B*100:.2f}%")
print(f"  Solve time:          {solve_B:.2f}s")
print(f"  Time limit:          300s")
print(f"────────────────────────────────────────────────────────────────")

if status_B in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
    open_B     = [j for j in J_idx if x_B[j].solution_value() > 0.5]
    unserved_B = [i for i in I_idx if u_B[i].solution_value() > 0.5]
    travel_B   = sum(
        d_B[i] * D[i, j] * y_B[i, j].solution_value()
        for (i, j) in valid_pairs_B
    )
    served_dem_B = sum(d_B) - sum(d_B[i] for i in unserved_B)

    print(f"Stations built: {len(open_B)} / {K}")
    print(f"Unserved:       {len(unserved_B)} / {num_I}")
    print(f"Travel cost:    {travel_B:,.2f} EV-vehicle·km")
    if served_dem_B > 0:
        print(f"Avg distance:   {travel_B/served_dem_B:.3f} km")

    rows_B = []
    for j in open_B:
        assigned_B   = [i for i in neighbors_of_j[j] if y_B[i, j].solution_value() > 0.5]
        stn_demand_B = sum(d_B[i] for i in assigned_B)
        avg_dist_B   = (sum(D[i, j] for i in assigned_B) / len(assigned_B)
                        if assigned_B else 0.0)
        rows_B.append({
            "site_id":     sites["site_id"].iloc[j],
            "lat":         sites["lat"].iloc[j],
            "lon":         sites["lon"].iloc[j],
            "n_assigned":  len(assigned_B),
            "stn_demand":  round(stn_demand_B, 1),
            "avg_dist_km": round(avg_dist_B, 3),
            "load_pct":    round(stn_demand_B / CAP * 100, 1)
        })

    solution_B_df = pd.DataFrame(rows_B)
    sites_A = set(solution_df["site_id"])
    sites_B_set = set(solution_B_df["site_id"])
    print(f"\n── Site Selection Comparison A vs B ─────────────────────────────")
    print(f"  Sites in both:      {len(sites_A & sites_B_set)} / {K}")
    print(f"  Only in Scenario A: {sorted(sites_A - sites_B_set)}")
    print(f"  Only in Scenario B: {sorted(sites_B_set - sites_A)}")
    print(f"────────────────────────────────────────────────────────────────")

    solution_B_df.to_csv("Mountain_View_CFLP_solution_B.csv", index=False)
    print("\nSaved to Mountain_View_CFLP_solution_B.csv")

Scenario A total demand: 5339.4 units
Scenario B total demand: 10518.1 units
Mean renter share: 0.522

Variables:   31868
Constraints: 31869
Solving (Scenario B)...

── Solver Audit (Scenario B) ────────────────────────────────────
  Variables:           31868 (all binary)
  Constraints:         31869
  Status:              Feasible (time limit)
  Objective value:     923,137.6969 EV-vehicle·km
  Best bound:          4,757.6065 EV-vehicle·km
  MIP gap:             99.48%
  Solve time:          298.90s
  Time limit:          300s
────────────────────────────────────────────────────────────────
Stations built: 20 / 20
Unserved:       9 / 79
Travel cost:    6,885.97 EV-vehicle·km
Avg distance:   0.704 km

── Site Selection Comparison A vs B ─────────────────────────────
  Sites in both:      4 / 20
  Only in Scenario A: ['J251', 'J276', 'J279', 'J355', 'J356', 'J438', 'J524', 'J548', 'J610', 'J64', 'J698', 'J707', 'J718', 'J79', 'J86', 'J97']
  Only in Scenario B: ['J164', 'J222', 'J240',

In [8]:
# ================================
# CELL 7: Greedy + Simulated Annealing Heuristic
# ================================
import numpy as np
import time
import math

SA_TIME  = 55
T_START  = 500.0
T_END    = 0.1
ALPHA    = 0.995

d_h     = result["demand_A"].values
num_I_h = len(d_h)
I_h     = list(range(num_I_h))
J_h     = list(range(num_J))

nb_i = {i: [j for j in J_h if D[i, j] <= MAX_DIST_KM] for i in I_h}
nb_j = {j: [i for i in I_h if D[i, j] <= MAX_DIST_KM] for j in J_h}
reachable_h = [i for i in I_h if nb_i[i]]

def assign_coverage(open_set):
    load       = {j: 0.0 for j in open_set}
    assignment = {}
    for i in sorted(reachable_h, key=lambda i: -d_h[i]):
        candidates = [j for j in nb_i[i] if j in open_set and load[j] + d_h[i] <= CAP]
        if not candidates:
            candidates = [j for j in nb_i[i] if j in open_set]
        if candidates:
            j = min(candidates, key=lambda j: D[i, j])
            assignment[i] = j
            load[j]      += d_h[i]
    return assignment, load

def objective_h(assignment):
    return sum(d_h[i] * D[i, assignment[i]] for i in assignment)

# Phase 1: Coverage-first greedy
print("Phase 1: Coverage-first greedy...")
t0        = time.time()
uncovered = set(reachable_h)
open_set  = set()
closed_set = set(J_h)

while len(open_set) < K and uncovered:
    best_j, best_cover = None, -1
    for j in closed_set:
        cover = sum(1 for i in nb_j[j] if i in uncovered)
        if cover > best_cover:
            best_cover, best_j = cover, j
    if best_j is None or best_cover == 0:
        break
    open_set.add(best_j)
    closed_set.remove(best_j)
    uncovered -= set(nb_j[best_j])

if len(open_set) < K:
    def density(j):
        nb = nb_j[j]
        return sum(d_h[i] for i in nb) / (np.mean([D[i, j] for i in nb]) + 1e-6) if nb else 0
    for j in sorted(closed_set, key=lambda j: -density(j)):
        if len(open_set) >= K:
            break
        open_set.add(j)
        closed_set.discard(j)

assignment, load = assign_coverage(open_set)
obj = objective_h(assignment)
print(f"  Covered: {len(assignment)}/{len(reachable_h)}  Obj: {obj:,.2f}  Time: {time.time()-t0:.2f}s")

# Phase 2: Simulated annealing
print("Phase 2: Simulated annealing...")
best_open, best_assignment, best_obj = set(open_set), dict(assignment), obj
current_open, current_assignment, current_obj = set(open_set), dict(assignment), obj
T, n_iter, n_accepted, n_improved = T_START, 0, 0, 0
closed_list = list(closed_set)
sa_start = time.time()

while time.time() - sa_start < SA_TIME:
    j_out = np.random.choice(list(current_open))
    j_in  = np.random.choice(closed_list)
    candidate = (current_open - {j_out}) | {j_in}
    new_assignment, _ = assign_coverage(candidate)
    if len(new_assignment) < len(reachable_h):
        T = max(T * ALPHA, T_END)
        n_iter += 1
        continue
    new_obj = objective_h(new_assignment)
    delta   = new_obj - current_obj
    if delta < 0 or np.random.random() < math.exp(-delta / T):
        current_open, current_assignment, current_obj = candidate, new_assignment, new_obj
        closed_list = [j for j in J_h if j not in current_open]
        n_accepted += 1
        if current_obj < best_obj:
            best_open, best_assignment, best_obj = set(current_open), dict(current_assignment), current_obj
            n_improved += 1
    T = max(T * ALPHA, T_END)
    n_iter += 1

total_time = time.time() - t0
print(f"  Iterations: {n_iter:,}  Accepted: {n_accepted:,}  Improved: {n_improved:,}")

print(f"\nHEURISTIC SOLUTION (Scenario A)")
print(f"  Solve time:     {total_time:.2f}s")
print(f"  Objective:      {best_obj:,.2f} EV-vehicle·km")
print(f"  Stations built: {len(best_open)} / {K}")
print(f"  Served:         {len(best_assignment)} / {len(reachable_h)} ({len(best_assignment)/len(reachable_h)*100:.1f}%)")

rows_h = []
for j in sorted(best_open):
    assigned   = [i for i in reachable_h if best_assignment.get(i) == j]
    stn_demand = sum(d_h[i] for i in assigned)
    avg_dist   = np.mean([D[i, j] for i in assigned]) if assigned else 0.0
    rows_h.append({
        "site_id":     sites["site_id"].iloc[j],
        "lat":         sites["lat"].iloc[j],
        "lon":         sites["lon"].iloc[j],
        "n_assigned":  len(assigned),
        "stn_demand":  round(stn_demand, 1),
        "avg_dist_km": round(avg_dist, 3),
        "load_pct":    round(stn_demand / CAP * 100, 1)
    })

heuristic_df = pd.DataFrame(rows_h)
served_demand_h = sum(d_h[i] for i in best_assignment)
print(f"\nAvg. assignment distance: {best_obj/served_demand_h:.3f} km")
print(f"Travel cost per served point: {best_obj/len(best_assignment):.2f} EV-vehicle·km")

heuristic_df.to_csv("Mountain_View_heuristic_solution.csv", index=False)
print("\nSaved to Mountain_View_heuristic_solution.csv")

Phase 1: Coverage-first greedy...
  Covered: 79/79  Obj: 8,319.41  Time: 0.01s
Phase 2: Simulated annealing...
  Iterations: 78,256  Accepted: 475  Improved: 105

HEURISTIC SOLUTION (Scenario A)
  Solve time:     55.01s
  Objective:      2,519.82 EV-vehicle·km
  Stations built: 20 / 20
  Served:         79 / 79 (100.0%)

Avg. assignment distance: 0.472 km
Travel cost per served point: 31.90 EV-vehicle·km

Saved to Mountain_View_heuristic_solution.csv


In [9]:
# ================================
# CELL 8: Existing network baseline
# ================================
import pandas as pd
import numpy as np

d_ex   = result["demand_A"].values
num_I  = len(d_ex)
I_lats = result["lat"].values
I_lons = result["lon"].values

def haversine_scalar(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

D_existing = np.array([
    [haversine_scalar(I_lats[i], I_lons[i], row["lat"], row["lon"])
     for _, row in existing.iterrows()]
    for i in range(num_I)
])

CAP_EXISTING = 500
assignment_existing = {}
load_existing       = {j: 0.0 for j in range(len(existing))}

for i in sorted(range(num_I), key=lambda i: -d_ex[i]):
    candidates = [
        j for j in range(len(existing))
        if D_existing[i, j] <= MAX_DIST_KM and load_existing[j] + d_ex[i] <= CAP_EXISTING
    ]
    if candidates:
        j = min(candidates, key=lambda j: D_existing[i, j])
        assignment_existing[i] = j
        load_existing[j]      += d_ex[i]

served_existing        = len(assignment_existing)
unserved_existing      = num_I - served_existing
served_demand_existing = sum(d_ex[i] for i in assignment_existing)
total_demand           = sum(d_ex)
travel_existing        = sum(
    d_ex[i] * D_existing[i, assignment_existing[i]]
    for i in assignment_existing
)

print(f"── Existing Network Evaluation ─────────────────────────────────")
print(f"  Block groups served:   {served_existing} / {num_I}")
print(f"  Block groups unserved: {unserved_existing} / {num_I}")
print(f"  Served demand:         {served_demand_existing:.1f} / {total_demand:.1f} units")
print(f"  Travel cost:           {travel_existing:,.2f} EV-vehicle·km")
if served_existing > 0:
    print(f"  Avg distance:          {travel_existing/served_demand_existing:.3f} km")
print(f"────────────────────────────────────────────────────────────────")

unserved_bg = [i for i in range(num_I) if i not in assignment_existing]
print(f"\nUnserved block group(s): {len(unserved_bg)}")
for i in unserved_bg:
    print(f"  GEOID: {result['block_group'].iloc[i]}")
    print(f"  Demand: {d_ex[i]:.1f} units")
    reachable_sites = sorted([
        (D[i, j], sites['site_id'].iloc[j])
        for j in range(len(sites)) if D[i, j] <= MAX_DIST_KM
    ])
    if reachable_sites:
        best_dist, best_sid = reachable_sites[0]
        print(f"  Best candidate site: {best_sid} ({best_dist:.3f} km)")

travel_plus1 = travel_existing + d_ex[unserved_bg[0]] * reachable_sites[0][0]
served_plus1 = served_existing + 1
dem_plus1    = served_demand_existing + d_ex[unserved_bg[0]]

print(f"\n── Three-Way Comparison ─────────────────────────────────────────")
print(f"  {'Metric':<35} {'Existing':>10} {'Exist+1':>10} {'MILP':>10}")
print(f"  {'-'*65}")
print(f"  {'Stations':<35} {'33':>10} {'34':>10} {'20':>10}")
print(f"  {'Block groups served':<35} {served_existing:>10} {served_plus1:>10} {79:>10}")
print(f"  {'Served demand (units)':<35} {served_demand_existing:>10.1f} {dem_plus1:>10.1f} {5339.4:>10.1f}")
print(f"  {'Travel cost (EV-veh·km)':<35} {travel_existing:>10.2f} {travel_plus1:>10.2f} {2513.26:>10.2f}")
print(f"  {'Avg distance (km)':<35} {travel_existing/served_demand_existing:>10.3f} {travel_plus1/dem_plus1:>10.3f} {0.471:>10.3f}")
print(f"  {'-'*65}")

── Existing Network Evaluation ─────────────────────────────────
  Block groups served:   78 / 79
  Block groups unserved: 1 / 79
  Served demand:         5289.4 / 5339.4 units
  Travel cost:           2,865.99 EV-vehicle·km
  Avg distance:          0.542 km
────────────────────────────────────────────────────────────────

Unserved block group(s): 1
  GEOID: 60855046011
  Demand: 50.0 units
  Best candidate site: J64 (1.381 km)

── Three-Way Comparison ─────────────────────────────────────────
  Metric                                Existing    Exist+1       MILP
  -----------------------------------------------------------------
  Stations                                    33         34         20
  Block groups served                         78         79         79
  Served demand (units)                   5289.4     5339.4     5339.4
  Travel cost (EV-veh·km)                2865.99    2935.03    2513.26
  Avg distance (km)                        0.542      0.550      0.471
  -----

In [10]:
# ================================
# CELL 9: Equity analysis (HCVI-stratified)
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import time

d_A = result["demand_A"].values

open_site_ids     = set(solution_df["site_id"])
open_site_indices = [sites[sites["site_id"] == sid].index[0] for sid in open_site_ids]

milp_assignment = {}
for i in range(len(result)):
    candidates = [j for j in open_site_indices if D[i, j] <= MAX_DIST_KM]
    if candidates:
        milp_assignment[i] = min(candidates, key=lambda j: D[i, j])

def equity_metrics(assignment, d, H_idx, label="Solution"):
    H_set     = set(H_idx)
    non_H     = [i for i in range(len(d)) if i not in H_set]
    served_H  = [i for i in H_idx if i in assignment]
    served_nH = [i for i in non_H if i in assignment]
    avg_dist_H  = np.mean([D[i, assignment[i]] for i in served_H]) if served_H else np.nan
    avg_dist_nH = np.mean([D[i, assignment[i]] for i in served_nH]) if served_nH else np.nan
    all_dists   = [D[i, assignment[i]] for i in assignment]
    d95         = np.percentile(all_dists, 95) if all_dists else np.nan
    coverage_gap = (len([i for i in H_idx if i not in assignment])/len(H_idx)
                    - len([i for i in non_H if i not in assignment])/len(non_H))
    dist_gap = avg_dist_H - avg_dist_nH

    print(f"\n── Equity Metrics: {label} ──────────────────────────────────")
    print(f"  {'Metric':<40} {'H (vuln)':>10} {'Non-H':>10}")
    print(f"  {'-'*60}")
    print(f"  {'Block groups':<40} {len(H_idx):>10} {len(non_H):>10}")
    print(f"  {'Served':<40} {len(served_H):>10} {len(served_nH):>10}")
    print(f"  {'Avg assignment distance (km)':<40} {avg_dist_H:>10.3f} {avg_dist_nH:>10.3f}")
    print(f"  {'-'*60}")
    print(f"  Coverage gap:   {coverage_gap:+.3f}")
    print(f"  Distance gap:   {dist_gap:+.3f} km")
    print(f"  D95:            {d95:.3f} km")
    print(f"────────────────────────────────────────────────────────────────")
    return {"served_H": len(served_H), "served_nonH": len(served_nH),
            "avg_dist_H": avg_dist_H, "avg_dist_nonH": avg_dist_nH,
            "coverage_gap": coverage_gap, "dist_gap": dist_gap, "D95": d95}

milp_eq = equity_metrics(milp_assignment, d_A, H_idx, "MILP (Scenario A)")

heu_open_ids     = set(heuristic_df["site_id"])
heu_open_indices = [sites[sites["site_id"] == sid].index[0] for sid in heu_open_ids]
heu_assignment   = {}
for i in range(len(result)):
    candidates = [j for j in heu_open_indices if D[i, j] <= MAX_DIST_KM]
    if candidates:
        heu_assignment[i] = min(candidates, key=lambda j: D[i, j])

heu_eq = equity_metrics(heu_assignment, d_A, H_idx, "Heuristic (Scenario A)")

# Equity-constrained MILP: vary delta
print(f"\n── Equity-Constrained MILP: Varying Delta ───────────────────────")
print(f"  Constraint: avg_dist(H) <= avg_dist(non-H) + delta")
print(f"  Unconstrained result: dist_gap = {milp_eq['dist_gap']:+.3f} km")
print(f"────────────────────────────────────────────────────────────────")

equity_results = []
for delta in [None, 0.2, 0.1, 0.0]:
    label = "Unconstrained" if delta is None else f"delta={delta}"
    solver_eq = pywraplp.Solver.CreateSolver("SCIP")
    solver_eq.SetTimeLimit(120_000)
    x_eq = [solver_eq.BoolVar(f"x_{j}") for j in J_idx]
    y_eq = {(i, j): solver_eq.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs}
    u_eq = [solver_eq.BoolVar(f"u_{i}") for i in I_idx]
    M_eq  = float(MAX_DIST_KM * max(d_A))
    obj_eq = solver_eq.Objective()
    for (i, j) in valid_pairs:
        obj_eq.SetCoefficient(y_eq[i, j], float(d_A[i]) * float(D[i, j]))
    for i in I_idx:
        obj_eq.SetCoefficient(u_eq[i], M_eq * float(d_A[i]))
    obj_eq.SetMinimization()
    for i in I_idx:
        if neighbors_of_i[i]:
            solver_eq.Add(sum(y_eq[i, j] for j in neighbors_of_i[i]) + u_eq[i] == 1)
        else:
            solver_eq.Add(u_eq[i] == 1)
    for (i, j) in valid_pairs:
        solver_eq.Add(y_eq[i, j] <= x_eq[j])
    solver_eq.Add(sum(x_eq[j] for j in J_idx) == K)
    for j in J_idx:
        if neighbors_of_j[j]:
            solver_eq.Add(sum(d_A[i] * y_eq[i, j] for i in neighbors_of_j[j]) <= CAP)
    if delta is not None:
        H_pairs    = [(i, j) for (i, j) in valid_pairs if i in H_set]
        nonH_pairs = [(i, j) for (i, j) in valid_pairs if i not in H_set]
        n_H    = len(H_idx)
        n_nonH = len(result) - n_H
        solver_eq.Add(
            n_nonH * sum(D[i, j] * y_eq[i, j] for (i, j) in H_pairs) -
            n_H    * sum(D[i, j] * y_eq[i, j] for (i, j) in nonH_pairs)
            <= delta * n_H * n_nonH
        )
    t0_eq    = time.time()
    status_eq = solver_eq.Solve()
    solve_eq  = time.time() - t0_eq
    if status_eq in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        assign_eq = {}
        for (i, j) in valid_pairs:
            if y_eq[i, j].solution_value() > 0.5:
                assign_eq[i] = j
        unserved_eq  = [i for i in I_idx if u_eq[i].solution_value() > 0.5]
        travel_eq    = sum(d_A[i] * D[i, j] * y_eq[i, j].solution_value() for (i, j) in valid_pairs)
        dist_H_eq    = np.mean([D[i, assign_eq[i]] for i in H_idx if i in assign_eq])
        dist_nonH_eq = np.mean([D[i, assign_eq[i]] for i in I_idx if i not in H_set and i in assign_eq])
        served_eq    = len(result) - len(unserved_eq)
        status_str   = "Optimal" if status_eq == pywraplp.Solver.OPTIMAL else "Feasible"
        print(f"\n  {label} ({status_str}, {solve_eq:.1f}s):")
        print(f"    Travel cost:    {travel_eq:,.2f} EV-vehicle·km")
        print(f"    Served:         {served_eq} / {len(result)}")
        print(f"    Avg dist H:     {dist_H_eq:.3f} km")
        print(f"    Avg dist non-H: {dist_nonH_eq:.3f} km")
        print(f"    Distance gap:   {dist_H_eq - dist_nonH_eq:+.3f} km")
        equity_results.append({
            "delta": label, "travel_cost": round(travel_eq, 2),
            "served": served_eq, "dist_H": round(dist_H_eq, 3),
            "dist_nonH": round(dist_nonH_eq, 3),
            "dist_gap": round(dist_H_eq - dist_nonH_eq, 3),
            "status": status_str, "solve_time": round(solve_eq, 1)
        })

eq_df = pd.DataFrame(equity_results)
print(f"\n── Equity Constraint Summary ────────────────────────────────────")
print(eq_df.to_string(index=False))
eq_df.to_csv("Mountain_View_equity_results.csv", index=False)
print("\nSaved to Mountain_View_equity_results.csv")


── Equity Metrics: MILP (Scenario A) ──────────────────────────────────
  Metric                                     H (vuln)      Non-H
  ------------------------------------------------------------
  Block groups                                     20         59
  Served                                           20         59
  Avg assignment distance (km)                  0.400      0.588
  ------------------------------------------------------------
  Coverage gap:   +0.000
  Distance gap:   -0.187 km
  D95:            1.178 km
────────────────────────────────────────────────────────────────

── Equity Metrics: Heuristic (Scenario A) ──────────────────────────────────
  Metric                                     H (vuln)      Non-H
  ------------------------------------------------------------
  Block groups                                     20         59
  Served                                           20         59
  Avg assignment distance (km)                  0.400      0

In [11]:
# ================================
# CELL 10: Road-network distance robustness check
# ================================
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd

print("Downloading Mountain View road network...")
G = ox.graph_from_place("Mountain View, California, USA", network_type="drive")
print(f"Graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

road_results = []
errors = 0
last_error = None

for i, j in milp_assignment.items():
    bg_lat   = result["lat"].iloc[i]
    bg_lon   = result["lon"].iloc[i]
    st_lat   = sites["lat"].iloc[j]
    st_lon   = sites["lon"].iloc[j]
    hav_dist = D[i, j]
    try:
        orig_node = ox.distance.nearest_nodes(G, bg_lon, bg_lat)
        dest_node = ox.distance.nearest_nodes(G, st_lon, st_lat)
        road_m    = nx.shortest_path_length(G, orig_node, dest_node, weight="length")
        road_km   = road_m / 1000.0
        ratio     = road_km / hav_dist if hav_dist > 0 else np.nan
        road_results.append({
            "block_group":  result["block_group"].iloc[i],
            "site_id":      sites["site_id"].iloc[j],
            "haversine_km": round(hav_dist, 4),
            "road_km":      round(road_km, 4),
            "ratio":        round(ratio, 4),
            "diff_km":      round(road_km - hav_dist, 4)
        })
    except Exception as e:
        errors += 1
        last_error = e

print(f"Computed: {len(road_results)} pairs  Errors: {errors}")
if last_error:
    print(f"Last error: {type(last_error).__name__}: {last_error}")

# Debug: test one pair manually
print("\nDebug: testing first pair manually...")
i0, j0 = list(milp_assignment.items())[0]
print(f"  Block group index: {i0}, site index: {j0}")
print(f"  BG coords: lat={result['lat'].iloc[i0]:.4f}, lon={result['lon'].iloc[i0]:.4f}")
print(f"  Site coords: lat={sites['lat'].iloc[j0]:.4f}, lon={sites['lon'].iloc[j0]:.4f}")
try:
    orig = ox.distance.nearest_nodes(G, result['lon'].iloc[i0], result['lat'].iloc[i0])
    dest = ox.distance.nearest_nodes(G, sites['lon'].iloc[j0], sites['lat'].iloc[j0])
    print(f"  Orig node: {orig}, Dest node: {dest}")
    length = nx.shortest_path_length(G, orig, dest, weight="length")
    print(f"  Road distance: {length/1000:.3f} km")
except Exception as e:
    print(f"  Error: {type(e).__name__}: {e}")

if len(road_results) > 0:
    df_road = pd.DataFrame(road_results)
else:
    print("\nFalling back to local CSV...")
    import os
    local_csv = "Mountain_View_road_vs_haversine.csv"
    if os.path.exists(local_csv):
        df_road = pd.read_csv(local_csv)
        print(f"Loaded {len(df_road)} pairs from local CSV")
    else:
        print("No local CSV found — skipping road network analysis")
        df_road = None

if df_road is not None:
    print(f"\n── Road-Network vs Haversine Distance Comparison ───────────────")
    print(f"  {'Metric':<40} {'Haversine':>12} {'Road network':>14}")
    print(f"  {'-'*66}")
    print(f"  {'Mean distance (km)':<40} {df_road['haversine_km'].mean():>12.3f} {df_road['road_km'].mean():>14.3f}")
    print(f"  {'Median distance (km)':<40} {df_road['haversine_km'].median():>12.3f} {df_road['road_km'].median():>14.3f}")
    print(f"  {'Max distance (km)':<40} {df_road['haversine_km'].max():>12.3f} {df_road['road_km'].max():>14.3f}")
    print(f"  {'Mean road/haversine ratio':<40} {df_road['ratio'].mean():>12.3f}")
    print(f"  {'Median road/haversine ratio':<40} {df_road['ratio'].median():>12.3f}")
    d95_hav  = df_road["haversine_km"].quantile(0.95)
    d95_road = df_road["road_km"].quantile(0.95)
    print(f"  {'D95 (km)':<40} {d95_hav:>12.3f} {d95_road:>14.3f}")
    print(f"────────────────────────────────────────────────────────────────")
    print(f"  Computed: {len(df_road)} pairs  Errors: {errors}")
    print(f"\nTop 5 pairs by road/haversine ratio:")
    print(df_road.nlargest(5, "ratio")[["block_group","site_id","haversine_km","road_km","ratio"]].to_string(index=False))
    df_road.to_csv("Mountain_View_road_vs_haversine.csv", index=False)
    print(f"\nSaved to Mountain_View_road_vs_haversine.csv")

Graph: 1583 nodes, 3700 edges
Computed: 77 pairs  Errors: 2
Last error: NetworkXNoPath: Node 5288162673 not reachable from 5288682193

Debug: testing first pair manually...
  Block group index: 0, site index: 355
  BG coords: lat=37.4129, lon=-122.0927
  Site coords: lat=37.4093, lon=-122.0979
  Orig node: 65465780, Dest node: 65464509
  Road distance: 0.775 km

── Road-Network vs Haversine Distance Comparison ───────────────
  Metric                                      Haversine   Road network
  ------------------------------------------------------------------
  Mean distance (km)                              0.538          0.949
  Median distance (km)                            0.495          0.659
  Max distance (km)                               2.057          6.915
  Mean road/haversine ratio                       1.681
  Median road/haversine ratio                     1.317
  D95 (km)                                        1.186          2.336
──────────────────────────────────

In [12]:
# ================================
# CELL 11: Sensitivity grid
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import time

def run_milp(d, K, CAP, max_dist, time_limit=60_000):
    num_I = len(d)
    num_J = len(sites)
    I_idx = range(num_I)
    J_idx = range(num_J)
    neighbors_i = {i: [j for j in J_idx if D[i, j] <= max_dist] for i in I_idx}
    neighbors_j = {j: [i for i in I_idx if D[i, j] <= max_dist] for j in J_idx}
    valid_pairs = [(i, j) for i in I_idx for j in neighbors_i[i]]
    solver = pywraplp.Solver.CreateSolver("SCIP")
    solver.SetTimeLimit(time_limit)
    x = [solver.BoolVar(f"x_{j}") for j in J_idx]
    y = {(i, j): solver.BoolVar(f"y_{i}_{j}") for (i, j) in valid_pairs}
    u = [solver.BoolVar(f"u_{i}") for i in I_idx]
    M = float(max_dist * max(d)) if len(d) > 0 else 1.0
    obj = solver.Objective()
    for (i, j) in valid_pairs:
        obj.SetCoefficient(y[i, j], float(d[i]) * float(D[i, j]))
    for i in I_idx:
        obj.SetCoefficient(u[i], M * float(d[i]))
    obj.SetMinimization()
    for i in I_idx:
        if neighbors_i[i]:
            solver.Add(sum(y[i, j] for j in neighbors_i[i]) + u[i] == 1)
        else:
            solver.Add(u[i] == 1)
    for (i, j) in valid_pairs:
        solver.Add(y[i, j] <= x[j])
    solver.Add(sum(x[j] for j in J_idx) == K)
    for j in J_idx:
        if neighbors_j[j]:
            solver.Add(sum(d[i] * y[i, j] for i in neighbors_j[j]) <= CAP)
    t0     = time.time()
    status = solver.Solve()
    solve_t = time.time() - t0
    if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        unserved_idx  = [i for i in I_idx if u[i].solution_value() > 0.5]
        served        = num_I - len(unserved_idx)
        travel        = sum(d[i] * D[i, j] * y[i, j].solution_value() for (i, j) in valid_pairs)
        served_demand = sum(d[i] for i in I_idx) - sum(d[i] for i in unserved_idx)
        avg_dist      = travel / served_demand if served_demand > 0 else 0
        avg_load      = served_demand / K if K > 0 else 0
        return {
            "served": served, "unserved": len(unserved_idx),
            "served_pct": round(served / num_I * 100, 1),
            "travel_cost": round(travel, 2), "avg_dist_km": round(avg_dist, 3),
            "avg_load": round(avg_load, 1), "load_pct": round(avg_load / CAP * 100, 1),
            "valid_pairs": len(valid_pairs),
            "status": "Optimal" if status == pywraplp.Solver.OPTIMAL else "Feasible",
            "solve_time": round(solve_t, 1)
        }
    return {"served": None, "unserved": None, "served_pct": None,
            "travel_cost": None, "avg_dist_km": None, "avg_load": None,
            "load_pct": None, "valid_pairs": len(valid_pairs),
            "status": "Infeasible", "solve_time": round(solve_t, 1)}

d_base        = result["demand_A"].values
vehicles_base = result["vehicles"].values

print("── Grid 1: Varying K ────────────────────────────────────────────")
rows_K = []
for K_val in [15, 20, 25, 30]:
    r = run_milp(d_base, K_val, 500, 3.0)
    r["K"] = K_val
    rows_K.append(r)
    print(f"  K={K_val}: served={r['served']}/79 ({r['served_pct']}%)  travel={r['travel_cost']}  avg_dist={r['avg_dist_km']} km  status={r['status']}")

print("\n── Grid 2: Varying Q ────────────────────────────────────────────")
rows_Q = []
for Q_val in [300, 500, 750]:
    r = run_milp(d_base, 20, Q_val, 3.0)
    r["Q"] = Q_val
    rows_Q.append(r)
    print(f"  Q={Q_val}: served={r['served']}/79 ({r['served_pct']}%)  load={r['load_pct']}%  status={r['status']}")

print("\n── Grid 3: Varying r ────────────────────────────────────────────")
rows_r = []
for r_val in [2.0, 3.0, 5.0]:
    r = run_milp(d_base, 20, 500, r_val)
    r["r_km"] = r_val
    rows_r.append(r)
    print(f"  r={r_val} km: pairs={r['valid_pairs']}  travel={r['travel_cost']}  status={r['status']}")

print("\n── Grid 4: Varying EV adoption rate ────────────────────────────")
rows_adopt = []
for rate in [0.05, 0.10, 0.20]:
    d_rate = vehicles_base * rate
    r = run_milp(d_rate, 20, 500, 3.0)
    r["adoption_pct"] = f"{int(rate*100)}%"
    r["total_demand"] = round(sum(d_rate), 1)
    rows_adopt.append(r)
    print(f"  adoption={int(rate*100)}%: demand={r['total_demand']}  served={r['served']}/79  status={r['status']}")

df_K     = pd.DataFrame(rows_K)[["K","served","served_pct","travel_cost","avg_dist_km","avg_load","load_pct","status"]]
df_Q     = pd.DataFrame(rows_Q)[["Q","served","served_pct","travel_cost","avg_dist_km","avg_load","load_pct","status"]]
df_r     = pd.DataFrame(rows_r)[["r_km","served","served_pct","travel_cost","avg_dist_km","valid_pairs","status"]]
df_adopt = pd.DataFrame(rows_adopt)[["adoption_pct","total_demand","served","served_pct","travel_cost","avg_dist_km","status"]]

df_K.to_csv("sensitivity_K.csv", index=False)
df_Q.to_csv("sensitivity_Q.csv", index=False)
df_r.to_csv("sensitivity_r.csv", index=False)
df_adopt.to_csv("sensitivity_adoption.csv", index=False)
print("\nSaved all sensitivity CSVs.")

── Grid 1: Varying K ────────────────────────────────────────────
  K=15: served=79/79 (100.0%)  travel=2925.37  avg_dist=0.548 km  status=Optimal
  K=20: served=79/79 (100.0%)  travel=2513.26  avg_dist=0.471 km  status=Optimal
  K=25: served=79/79 (100.0%)  travel=2211.47  avg_dist=0.414 km  status=Optimal
  K=30: served=79/79 (100.0%)  travel=2011.99  avg_dist=0.377 km  status=Optimal

── Grid 2: Varying Q ────────────────────────────────────────────
  Q=300: served=79/79 (100.0%)  load=89.0%  status=Feasible
  Q=500: served=79/79 (100.0%)  load=53.4%  status=Optimal
  Q=750: served=79/79 (100.0%)  load=35.6%  status=Optimal

── Grid 3: Varying r ────────────────────────────────────────────
  r=2.0 km: pairs=16488  travel=2517.23  status=Optimal
  r=3.0 km: pairs=30997  travel=2513.26  status=Optimal
  r=5.0 km: pairs=53167  travel=2513.26  status=Optimal

── Grid 4: Varying EV adoption rate ────────────────────────────
  adoption=5%: demand=2669.7  served=79/79  status=Optimal
  ado

In [13]:
# ================================
# CELL 12: Scenario-based stochastic demand analysis
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import time

vehicles     = result["vehicles"].values
renter_share = result["renter_share"].values
d_A          = result["demand_A"].values
d_B          = result["demand_B"].values
hcvi_norm    = result_hcvi["HCVI_norm"].values

DOWNTOWN_LAT = 37.3861
DOWNTOWN_LON = -122.0839

def haversine_scalar(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

dist_to_downtown = np.array([
    haversine_scalar(row["lat"], row["lon"], DOWNTOWN_LAT, DOWNTOWN_LON)
    for _, row in result.iterrows()
])
commute_weight = 1 / (dist_to_downtown + 0.5)
commute_weight = commute_weight / commute_weight.mean()

scenarios = {
    "S1_baseline":        {"demand": vehicles * 0.10,         "label": "Baseline (10% uniform stock)"},
    "S2_high_adoption":   {"demand": np.minimum(vehicles * 0.20, 500), "label": "High adoption (20% stock)"},
    "S3_tenure_weighted": {"demand": d_B,                     "label": "Tenure-weighted need (Scenario B)"},
    "S4_apartment_heavy": {"demand": (vehicles*(1-renter_share)*0.03 + vehicles*renter_share*0.50), "label": "Apartment-heavy"},
    "S5_equity_growth":   {"demand": vehicles*0.10*(1+hcvi_norm), "label": "Equity-growth (HCVI-weighted)"},
    "S6_commute_heavy":   {"demand": vehicles*0.10*commute_weight*renter_share, "label": "Commute-heavy"},
}

def solve_scenario_sc(d_vec, K=20, CAP=500, max_dist=3.0, time_limit=60_000):
    d_vec = np.minimum(d_vec, CAP * 0.95)
    num_I = len(d_vec)
    num_J = len(sites)
    I_idx = range(num_I)
    J_idx = range(num_J)
    nb_i = {i: [j for j in J_idx if D[i,j] <= max_dist] for i in I_idx}
    nb_j = {j: [i for i in I_idx if D[i,j] <= max_dist] for j in J_idx}
    vp   = [(i,j) for i in I_idx for j in nb_i[i]]
    solver = pywraplp.Solver.CreateSolver("SCIP")
    solver.SetTimeLimit(time_limit)
    x = [solver.BoolVar(f"x_{j}") for j in J_idx]
    y = {(i,j): solver.BoolVar(f"y_{i}_{j}") for (i,j) in vp}
    u = [solver.BoolVar(f"u_{i}") for i in I_idx]
    M = float(max_dist * max(d_vec)) if len(d_vec) > 0 else 1.0
    obj = solver.Objective()
    for (i,j) in vp:
        obj.SetCoefficient(y[i,j], float(d_vec[i]) * float(D[i,j]))
    for i in I_idx:
        obj.SetCoefficient(u[i], M * float(d_vec[i]))
    obj.SetMinimization()
    for i in I_idx:
        if nb_i[i]:
            solver.Add(sum(y[i,j] for j in nb_i[i]) + u[i] == 1)
        else:
            solver.Add(u[i] == 1)
    for (i,j) in vp:
        solver.Add(y[i,j] <= x[j])
    solver.Add(sum(x[j] for j in J_idx) == K)
    for j in J_idx:
        if nb_j[j]:
            solver.Add(sum(d_vec[i] * y[i,j] for i in nb_j[j]) <= CAP)
    t0 = time.time()
    status = solver.Solve()
    solve_t = time.time() - t0
    if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        open_sites = [j for j in J_idx if x[j].solution_value() > 0.5]
        unserved   = [i for i in I_idx if u[i].solution_value() > 0.5]
        travel     = sum(d_vec[i] * D[i,j] * y[i,j].solution_value() for (i,j) in vp)
        served_dem = sum(d_vec) - sum(d_vec[i] for i in unserved)
        return {
            "status":      "Optimal" if status == pywraplp.Solver.OPTIMAL else "Feasible",
            "open_sites":  set(open_sites),
            "site_ids":    set(sites["site_id"].iloc[j] for j in open_sites),
            "unserved":    len(unserved),
            "served":      len(I_idx) - len(unserved),
            "travel_cost": travel,
            "served_dem":  served_dem,
            "avg_dist":    travel / served_dem if served_dem > 0 else 0,
            "solve_time":  solve_t
        }
    return None

def evaluate_fixed(open_idx, d_vec, CAP=500, max_dist=3.0):
    d_vec = np.minimum(d_vec, CAP * 0.95)
    assignment = {}
    load = {j: 0.0 for j in open_idx}
    for i in sorted(range(len(d_vec)), key=lambda i: -d_vec[i]):
        candidates = [j for j in open_idx if D[i,j] <= max_dist and load[j] + d_vec[i] <= CAP]
        if candidates:
            j = min(candidates, key=lambda j: D[i,j])
            assignment[i] = j
            load[j] += d_vec[i]
    travel     = sum(d_vec[i] * D[i, assignment[i]] for i in assignment)
    served_dem = sum(d_vec[i] for i in assignment)
    return {
        "served": len(assignment), "unserved": len(d_vec) - len(assignment),
        "travel_cost": travel, "served_dem": served_dem,
        "avg_dist": travel / served_dem if served_dem > 0 else 0
    }

print("── Solving Best Solution for Each Scenario ──────────────────────")
best_solutions = {}
for s_id, s in scenarios.items():
    print(f"  Solving {s['label']}...", end=" ", flush=True)
    res = solve_scenario_sc(s["demand"])
    if res:
        best_solutions[s_id] = res
        print(f"travel={res['travel_cost']:,.1f}  served={res['served']}/79  status={res['status']}  time={res['solve_time']:.1f}s")
    else:
        print("FAILED")

print("\n── Evaluating Scenario A Solution Across All Scenarios ──────────")
A_open_idx = xA_open_indices  # stored in Cell 5
eval_results = {}
for s_id, s in scenarios.items():
    d_vec = np.minimum(s["demand"], 500 * 0.95)
    ev = evaluate_fixed(A_open_idx, d_vec)
    eval_results[s_id] = ev
    print(f"  {s['label']:<35} travel={ev['travel_cost']:>10,.1f}  served={ev['served']}/79")

print("\n── Regret Analysis ──────────────────────────────────────────────")
print(f"  {'Scenario':<35} {'Best*':>10} {'ScenA eval':>12} {'Regret':>10} {'Regret%':>8}")
print(f"  {'-'*75}")
regrets = {}
for s_id, s in scenarios.items():
    if s_id in best_solutions and best_solutions[s_id]:
        best_cost  = best_solutions[s_id]["travel_cost"]
        eval_cost  = eval_results[s_id]["travel_cost"]
        regret     = eval_cost - best_cost
        regret_pct = regret / best_cost * 100 if best_cost > 0 else 0
        regrets[s_id] = regret
        print(f"  {s['label']:<35} {best_cost:>10,.1f} {eval_cost:>12,.1f} {regret:>10,.1f} {regret_pct:>7.1f}%")

max_regret_id  = max(regrets, key=regrets.get)
max_regret_val = regrets[max_regret_id]
print(f"\n  Max regret: {scenarios[max_regret_id]['label']} ({max_regret_val:,.1f} EV-vehicle·km)")

travel_costs = [eval_results[s_id]["travel_cost"] for s_id in scenarios]
travel_costs_arr = np.array(sorted(travel_costs))
alpha   = 0.90
n_scen  = len(travel_costs_arr)
cutoff  = int(np.ceil(alpha * n_scen))
cvar_90 = travel_costs_arr[cutoff-1:].mean()
exp_cost = travel_costs_arr.mean()

print(f"\n── Risk Metrics ─────────────────────────────────────────────────")
print(f"  Expected travel cost (avg across scenarios): {exp_cost:,.2f}")
print(f"  CVaR_0.90 (avg of top 10% worst scenarios):  {cvar_90:,.2f}")
print(f"────────────────────────────────────────────────────────────────")

print("\n── Site Selection Stability ─────────────────────────────────────")
site_freq = {}
for s_id, sol in best_solutions.items():
    for sid in sol["site_ids"]:
        site_freq[sid] = site_freq.get(sid, 0) + 1

stable_core   = {sid for sid, freq in site_freq.items() if freq >= 5}
moderate      = {sid for sid, freq in site_freq.items() if freq in (3, 4)}
scenario_spec = {sid for sid, freq in site_freq.items() if freq <= 2}
A_in_core     = xA_open_ids & stable_core

print(f"  Stable core (>=5/6 scenarios):      {len(stable_core)} sites: {sorted(stable_core)}")
print(f"  Moderate (3-4/6 scenarios):          {len(moderate)} sites")
print(f"  Scenario-specific (<=2/6 scenarios): {len(scenario_spec)} sites")
print(f"  Scenario A sites in stable core:     {len(A_in_core)}/20 -> {sorted(A_in_core)}")
print(f"────────────────────────────────────────────────────────────────")

rows_sc = []
for s_id, s in scenarios.items():
    best = best_solutions.get(s_id, {})
    ev   = eval_results[s_id]
    reg  = regrets.get(s_id, None)
    rows_sc.append({
        "Scenario":          s["label"],
        "Best* travel cost": round(best.get("travel_cost", float("nan")), 1),
        "Best* served":      best.get("served", "—"),
        "ScenA travel cost": round(ev["travel_cost"], 1),
        "ScenA served":      ev["served"],
        "Regret":            round(reg, 1) if reg is not None else "—",
        "Best* status":      best.get("status", "—"),
    })

summary_df = pd.DataFrame(rows_sc)
print("\n── Full Scenario Summary Table ──────────────────────────────────")
print(summary_df.to_string(index=False))

summary_df.to_csv("Mountain_View_scenario_analysis.csv", index=False)
print("\nSaved to Mountain_View_scenario_analysis.csv")

── Solving Best Solution for Each Scenario ──────────────────────
  Solving Baseline (10% uniform stock)... travel=2,513.3  served=79/79  status=Optimal  time=4.5s
  Solving High adoption (20% stock)... travel=7,357.1  served=69/79  status=Feasible  time=60.0s
  Solving Tenure-weighted need (Scenario B)... travel=5,952.6  served=71/79  status=Feasible  time=60.0s
  Solving Apartment-heavy... travel=8,887.3  served=46/79  status=Feasible  time=60.1s
  Solving Equity-growth (HCVI-weighted)... travel=5,299.4  served=79/79  status=Feasible  time=60.0s
  Solving Commute-heavy... travel=1,116.6  served=78/79  status=Optimal  time=3.9s

── Evaluating Scenario A Solution Across All Scenarios ──────────
  Baseline (10% uniform stock)        travel=   2,513.3  served=79/79
  High adoption (20% stock)           travel=   8,076.8  served=65/79
  Tenure-weighted need (Scenario B)   travel=   9,160.8  served=68/79
  Apartment-heavy                     travel=  10,566.0  served=41/79
  Equity-growth 

In [14]:
# ================================
# CELL 13: Queueing / Wait-Time Model (M/M/m Erlang C)
# ================================
import numpy as np
import pandas as pd
from math import factorial

# ── Parameters ──────────────────────────────────────────────────────────────
M_PORTS      = 6        # ports per station (uniform baseline)
MU           = 1.0      # service rate: sessions per port per hour (60-min avg session)
PUBLIC_CHARGING_SHARE = 0.25  # fraction of assigned EVs that regularly use public charging
WEEKLY_VISITS = 2             # visits per week for those who do use public charging
PEAK_FACTOR  = 3.0      # peak hour is 3x average hourly demand
HOURS_PER_WEEK = 168    # 24 * 7

# ── Erlang C formula ────────────────────────────────────────────────────────
def erlang_c(m, rho_total):
    """
    Erlang C: P(wait > 0) for M/M/m queue.
    m         = number of servers (ports)
    rho_total = total offered load = lambda / mu (in Erlangs)
    rho_per_server = rho_total / m must be < 1 for stability
    """
    rho = rho_total / m  # per-server utilization
    if rho >= 1.0:
        return 1.0  # unstable — queue grows without bound

    # Numerator: (rho_total^m / m!) * (1 / (1 - rho))
    numerator = (rho_total ** m / factorial(m)) * (1 / (1 - rho))

    # Denominator: sum_{k=0}^{m-1} rho_total^k / k! + numerator
    sum_terms = sum(rho_total ** k / factorial(k) for k in range(m))
    denominator = sum_terms + numerator

    return numerator / denominator

def expected_wait(m, lam, mu):
    """
    Expected wait time in queue (hours) for M/M/m.
    W_q = C(m, rho) / (m * mu - lambda)
    Returns inf if unstable.
    """
    rho_total = lam / mu
    rho = rho_total / m
    if rho >= 1.0:
        return float("inf")
    c = erlang_c(m, rho_total)
    return c / (m * mu - lam)

# ── Compute arrival rates from MILP solution ─────────────────────────────────
# lambda_j = stn_demand * (weekly_visits / hours_per_week) * peak_factor
solution_df["lambda_j"] = (
    solution_df["stn_demand"] *
    PUBLIC_CHARGING_SHARE *
    (WEEKLY_VISITS / HOURS_PER_WEEK) *
    PEAK_FACTOR
)

# ── Baseline: m = 6 ports ────────────────────────────────────────────────────
rows_q = []
for _, row in solution_df.iterrows():
    lam  = row["lambda_j"]
    rho_total = lam / MU
    rho  = rho_total / M_PORTS
    pw   = erlang_c(M_PORTS, rho_total)
    wq   = expected_wait(M_PORTS, lam, MU)
    wq_min = wq * 60 if wq != float("inf") else float("inf")
    rows_q.append({
        "site_id":        row["site_id"],
        "stn_demand":     row["stn_demand"],
        "lambda_j":       round(lam, 4),
        "rho_total":      round(rho_total, 4),
        "rho_per_port":   round(rho, 4),
        "P_wait":         round(pw, 4),
        "W_q_hours":      round(wq, 4) if wq != float("inf") else "inf",
        "W_q_minutes":    round(wq_min, 2) if wq_min != float("inf") else "inf",
        "overload_risk":  rho >= 1.0
    })

df_queue = pd.DataFrame(rows_q)

print(f"── Queueing Model: M/M/m Erlang C ─────────────────────────────────")
print(f"  Ports per station (m):     {M_PORTS}")
print(f"  Service rate (μ):          {MU} sessions/port/hour (60-min avg)")
print(f"  Weekly visits per EV:      {WEEKLY_VISITS}")
print(f"  Peak factor:               {PEAK_FACTOR}×")
print(f"────────────────────────────────────────────────────────────────────")
print(f"\n── Station-Level Results (m={M_PORTS} ports) ──────────────────────────")
print(df_queue[["site_id","stn_demand","lambda_j","rho_per_port",
                "P_wait","W_q_minutes","overload_risk"]].to_string(index=False))

print(f"\n── Summary Statistics ───────────────────────────────────────────────")
print(f"  Mean utilization (ρ/port):     {df_queue['rho_per_port'].mean():.4f}")
print(f"  Max utilization (ρ/port):      {df_queue['rho_per_port'].max():.4f}")
print(f"  Mean P(wait):                  {df_queue['P_wait'].mean():.4f}")
print(f"  Max P(wait):                   {df_queue['P_wait'].max():.4f}")
finite_wq = df_queue[df_queue["W_q_minutes"] != "inf"]["W_q_minutes"].astype(float)
print(f"  Mean E[wait] (min):            {finite_wq.mean():.2f}")
print(f"  Max E[wait] (min):             {finite_wq.max():.2f}")
print(f"  Stations with overload risk:   {df_queue['overload_risk'].sum()}")
print(f"────────────────────────────────────────────────────────────────────")

# ── Sensitivity: vary m ───────────────────────────────────────────────────────
print(f"\n── Sensitivity: Varying Ports per Station ───────────────────────────")
print(f"  {'m':>4} {'Mean ρ':>8} {'Max ρ':>8} {'Mean P(wait)':>14} "
      f"{'Max P(wait)':>12} {'Mean Wq (min)':>14} {'Overloaded':>11}")
print(f"  {'-'*75}")

sensitivity_rows = []
for m_val in [4, 6, 8]:
    rhos, pws, wqs, overloads = [], [], [], []
    for _, row in solution_df.iterrows():
        lam = row["lambda_j"]
        rho_total = lam / MU
        rho = rho_total / m_val
        pw  = erlang_c(m_val, rho_total)
        wq  = expected_wait(m_val, lam, MU)
        rhos.append(rho)
        pws.append(pw)
        wqs.append(wq * 60 if wq != float("inf") else None)
        overloads.append(rho >= 1.0)
    finite_wqs = [w for w in wqs if w is not None]
    mean_wq    = np.mean(finite_wqs) if finite_wqs else float("inf")
    print(f"  {m_val:>4} {np.mean(rhos):>8.4f} {np.max(rhos):>8.4f} "
          f"{np.mean(pws):>14.4f} {np.max(pws):>12.4f} "
          f"{mean_wq:>14.2f} {sum(overloads):>11}")
    sensitivity_rows.append({
        "m": m_val,
        "mean_rho": round(np.mean(rhos), 4),
        "max_rho": round(np.max(rhos), 4),
        "mean_P_wait": round(np.mean(pws), 4),
        "max_P_wait": round(np.max(pws), 4),
        "mean_Wq_min": round(mean_wq, 2),
        "overloaded_stations": sum(overloads)
    })

df_sensitivity_q = pd.DataFrame(sensitivity_rows)

# ── Scenario B queueing stress test ──────────────────────────────────────────
print(f"\n── Scenario B Queueing Stress Test (tenure-weighted demand) ─────────")
print(f"  Using Scenario B station assignments (solution_B_df)")

rows_qB = []
for _, row in solution_B_df.iterrows():
    lam_B = row["stn_demand"] * (WEEKLY_VISITS / HOURS_PER_WEEK) * PEAK_FACTOR
    rho_total_B = lam_B / MU
    rho_B = rho_total_B / M_PORTS
    pw_B  = erlang_c(M_PORTS, rho_total_B)
    wq_B  = expected_wait(M_PORTS, lam_B, MU)
    wq_min_B = wq_B * 60 if wq_B != float("inf") else float("inf")
    rows_qB.append({
        "site_id":      row["site_id"],
        "stn_demand":   row["stn_demand"],
        "lambda_j":     round(lam_B, 4),
        "rho_per_port": round(rho_B, 4),
        "P_wait":       round(pw_B, 4),
        "W_q_minutes":  round(wq_min_B, 2) if wq_min_B != float("inf") else "inf",
        "overload_risk": rho_B >= 1.0
    })

df_queue_B = pd.DataFrame(rows_qB)
finite_B = df_queue_B[df_queue_B["W_q_minutes"] != "inf"]["W_q_minutes"].astype(float)

print(f"\n  Scenario A vs B queueing comparison (m={M_PORTS} ports):")
print(f"  {'Metric':<35} {'Scenario A':>12} {'Scenario B':>12}")
print(f"  {'-'*59}")
print(f"  {'Mean utilization (ρ/port)':<35} "
      f"{df_queue['rho_per_port'].mean():>12.4f} "
      f"{df_queue_B['rho_per_port'].mean():>12.4f}")
print(f"  {'Max utilization (ρ/port)':<35} "
      f"{df_queue['rho_per_port'].max():>12.4f} "
      f"{df_queue_B['rho_per_port'].max():>12.4f}")
print(f"  {'Mean P(wait)':<35} "
      f"{df_queue['P_wait'].mean():>12.4f} "
      f"{df_queue_B['P_wait'].mean():>12.4f}")
print(f"  {'Max P(wait)':<35} "
      f"{df_queue['P_wait'].max():>12.4f} "
      f"{df_queue_B['P_wait'].max():>12.4f}")
print(f"  {'Mean E[wait] (min)':<35} "
      f"{finite_wq.mean():>12.2f} "
      f"{finite_B.mean() if len(finite_B) > 0 else float('inf'):>12.2f}")
print(f"  {'Stations with overload risk':<35} "
      f"{df_queue['overload_risk'].sum():>12} "
      f"{df_queue_B['overload_risk'].sum():>12}")
print(f"────────────────────────────────────────────────────────────────────")

# ── Save ─────────────────────────────────────────────────────────────────────
df_queue.to_csv("Mountain_View_queueing_results.csv", index=False)
df_sensitivity_q.to_csv("queueing_sensitivity_m.csv", index=False)
df_queue_B.to_csv("Mountain_View_queueing_results_B.csv", index=False)
print("\nSaved Mountain_View_queueing_results.csv")
print("Saved queueing_sensitivity_m.csv")
print("Saved Mountain_View_queueing_results_B.csv")

── Queueing Model: M/M/m Erlang C ─────────────────────────────────
  Ports per station (m):     6
  Service rate (μ):          1.0 sessions/port/hour (60-min avg)
  Weekly visits per EV:      2
  Peak factor:               3.0×
────────────────────────────────────────────────────────────────────

── Station-Level Results (m=6 ports) ──────────────────────────
site_id  stn_demand  lambda_j  rho_per_port  P_wait  W_q_minutes  overload_risk
    J27       328.5    2.9330        0.4888  0.0907         1.78          False
    J64        50.0    0.4464        0.0744  0.0000         0.00          False
    J79       195.2    1.7429        0.2905  0.0096         0.14          False
    J86       224.3    2.0027        0.3338  0.0181         0.27          False
    J97        98.4    0.8786        0.1464  0.0003         0.00          False
   J251       235.1    2.0991        0.3499  0.0224         0.34          False
   J276       267.6    2.3893        0.3982  0.0392         0.65          Fal

In [15]:
# ================================
# CELL 14: Reliability / Single-Station Outage Analysis
# ================================
import numpy as np
import pandas as pd

# ── Setup ────────────────────────────────────────────────────────────────────
d_A          = result["demand_A"].values
open_indices = sorted(list(xA_open_indices))   # from Cell 5
num_I        = len(result)

def evaluate_without(removed_j, open_indices, d, D, MAX_DIST_KM, CAP):
    """
    Evaluate network with station removed_j taken offline.
    Greedily reassigns block groups to nearest remaining open station.
    Returns dict of coverage metrics.
    """
    remaining = [j for j in open_indices if j != removed_j]
    assignment = {}
    load = {j: 0.0 for j in remaining}

    for i in sorted(range(num_I), key=lambda i: -d[i]):
        candidates = [
            j for j in remaining
            if D[i, j] <= MAX_DIST_KM and load[j] + d[i] <= CAP
        ]
        if candidates:
            j = min(candidates, key=lambda j: D[i, j])
            assignment[i] = j
            load[j] += d[i]

    served        = len(assignment)
    unserved_idx  = [i for i in range(num_I) if i not in assignment]
    unserved_dem  = sum(d[i] for i in unserved_idx)
    travel        = sum(d[i] * D[i, assignment[i]] for i in assignment)
    served_dem    = sum(d[i] for i in assignment)
    avg_dist      = travel / served_dem if served_dem > 0 else 0

    return {
        "served":         served,
        "unserved":       len(unserved_idx),
        "unserved_idx":   unserved_idx,
        "unserved_demand": round(unserved_dem, 1),
        "travel_cost":    round(travel, 2),
        "avg_dist_km":    round(avg_dist, 3),
    }

# ── Baseline (no outage) ─────────────────────────────────────────────────────
baseline = evaluate_without(-1, open_indices + [-1], d_A, D, MAX_DIST_KM, CAP)
# Simpler: just use known values
baseline_served       = 79
baseline_travel       = 2513.26
baseline_unserved_dem = 0.0

# ── Single-station outage analysis ───────────────────────────────────────────
print("── Single-Station Outage Analysis ──────────────────────────────────")
print(f"  Baseline: {baseline_served}/79 served, travel={baseline_travel} EV-veh·km")
print(f"  Evaluating {len(open_indices)} single-station failure scenarios...\n")

rows_rel = []
for j in open_indices:
    site_id    = sites["site_id"].iloc[j]
    stn_demand = solution_df[solution_df["site_id"] == site_id]["stn_demand"].values[0]
    res        = evaluate_without(j, open_indices, d_A, D, MAX_DIST_KM, CAP)

    # Check if unserved block groups have any backup within 3km
    unserved_with_backup = []
    unserved_no_backup   = []
    for i in res["unserved_idx"]:
        remaining = [jj for jj in open_indices if jj != j]
        has_backup = any(D[i, jj] <= MAX_DIST_KM for jj in remaining)
        if has_backup:
            unserved_with_backup.append(i)
        else:
            unserved_no_backup.append(i)

    rows_rel.append({
        "site_id":              site_id,
        "stn_demand":           stn_demand,
        "bg_served_baseline":   79,
        "bg_unserved_outage":   res["unserved"],
        "demand_at_risk":       round(stn_demand, 1),
        "unserved_demand":      res["unserved_demand"],
        "travel_cost_outage":   res["travel_cost"],
        "travel_cost_delta":    round(res["travel_cost"] - baseline_travel, 2),
        "avg_dist_outage":      res["avg_dist_km"],
        "bg_no_backup":         len(unserved_no_backup),
        "bg_with_backup":       len(unserved_with_backup),
        "is_critical":          len(unserved_no_backup) > 0,
    })

df_rel = pd.DataFrame(rows_rel).sort_values("bg_unserved_outage", ascending=False)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"  {'Station':<8} {'Demand':>8} {'BG lost':>8} {'No backup':>10} "
      f"{'Unserved dem':>13} {'Travel delta':>13} {'Critical':>9}")
print(f"  {'-'*75}")
for _, row in df_rel.iterrows():
    print(f"  {row['site_id']:<8} {row['stn_demand']:>8.1f} "
          f"{row['bg_unserved_outage']:>8} {row['bg_no_backup']:>10} "
          f"{row['unserved_demand']:>13.1f} {row['travel_cost_delta']:>+13.2f} "
          f"{'YES' if row['is_critical'] else 'no':>9}")

# ── Key metrics ───────────────────────────────────────────────────────────────
worst      = df_rel.iloc[0]
n_critical = df_rel["is_critical"].sum()
n_zero     = (df_rel["bg_unserved_outage"] == 0).sum()

print(f"\n── Reliability Summary ──────────────────────────────────────────────")
print(f"  Stations where outage leaves 0 block groups unserved: {n_zero}/20")
print(f"  Stations with at least 1 block group with no backup:  {n_critical}/20")
print(f"\n  Worst-case single failure: {worst['site_id']}")
print(f"    Demand at risk:           {worst['stn_demand']:.1f} EV-vehicle units")
print(f"    Block groups losing service: {worst['bg_unserved_outage']}")
print(f"    Block groups with no backup: {worst['bg_no_backup']}")
print(f"    Unserved demand:          {worst['unserved_demand']:.1f} units")
print(f"    Travel cost increase:     {worst['travel_cost_delta']:+.2f} EV-vehicle·km")

# ── Redundancy score ──────────────────────────────────────────────────────────
# For each block group: how many open stations are within 3km?
redundancy = []
for i in range(num_I):
    n_backup = sum(1 for j in open_indices if D[i, j] <= MAX_DIST_KM)
    redundancy.append(n_backup)

redundancy = np.array(redundancy)
pct_single_coverage = (redundancy == 1).sum() / num_I * 100
pct_multi_coverage   = (redundancy >= 2).sum() / num_I * 100

print(f"\n── Network Redundancy ───────────────────────────────────────────────")
print(f"  Block groups with exactly 1 open station within 3km: "
      f"{(redundancy == 1).sum()} ({pct_single_coverage:.1f}%)")
print(f"  Block groups with ≥2 open stations within 3km:       "
      f"{(redundancy >= 2).sum()} ({pct_multi_coverage:.1f}%)")
print(f"  Block groups with 0 open stations within 3km:         "
      f"{(redundancy == 0).sum()}")
print(f"  Mean open stations within 3km per block group:        "
      f"{redundancy.mean():.2f}")
print(f"────────────────────────────────────────────────────────────────────")

# ── Save ──────────────────────────────────────────────────────────────────────
df_rel.to_csv("Mountain_View_reliability_outage.csv", index=False)
print("\nSaved Mountain_View_reliability_outage.csv")

── Single-Station Outage Analysis ──────────────────────────────────
  Baseline: 79/79 served, travel=2513.26 EV-veh·km
  Evaluating 20 single-station failure scenarios...

  Station    Demand  BG lost  No backup  Unserved dem  Travel delta  Critical
  ---------------------------------------------------------------------------
  J27         328.5        0          0           0.0       +202.10        no
  J64          50.0        0          0           0.0        +78.66        no
  J707        328.2        0          0           0.0       +135.01        no
  J698        269.1        0          0           0.0       +164.55        no
  J642        345.7        0          0           0.0       +294.49        no
  J610        347.1        0          0           0.0       +207.64        no
  J548        244.5        0          0           0.0       +148.61        no
  J524        237.4        0          0           0.0       +135.45        no
  J446        317.0        0          0        

In [16]:
# ================================
# CELL 15: Minimax-regret analysis
# ================================
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import time

# Use only S1 and S3 — the two most policy-relevant scenarios
# (S1: baseline demand, S3: tenure-weighted public-charging need)
# Full 5-scenario minimax exceeds solver budget at this scale
sc_ids_mm = ["S1_baseline", "S3_tenure_weighted"]

# Build demand vectors from scenario dict (already capped in Cell 12)
sc_demands = {
    s_id: np.minimum(scenarios[s_id]["demand"], CAP * 0.95)
    for s_id in sc_ids_mm
}

# Recover best* costs from Cell 12 best_solutions
scenario_best_mm  = {s_id: best_solutions[s_id]["travel_cost"] for s_id in sc_ids_mm}
scenario_xA_cost  = {s_id: eval_results[s_id]["travel_cost"]   for s_id in sc_ids_mm}

K_mm        = 20
CAP_mm      = 500
r_mm        = MAX_DIST_KM
num_I_mm    = len(result)
num_J_mm    = len(sites)
I_idx_mm    = range(num_I_mm)
J_idx_mm    = range(num_J_mm)
nb_i_mm     = {i: [j for j in J_idx_mm if D[i, j] <= r_mm] for i in I_idx_mm}
nb_j_mm     = {j: [i for i in I_idx_mm if D[i, j] <= r_mm] for j in J_idx_mm}
vp_mm       = [(i, j) for i in I_idx_mm for j in nb_i_mm[i]]

# ── Solve minimax-regret MILP ────────────────────────────────────────────────
# Linearization: minimize t  s.t.  t >= obj_s(x) - best*_s  for all s
# Variables: x_j (shared siting), t (max regret), y^s_ij and u^s_i per scenario
print("── Solving minimax-regret MILP ─────────────────────────────────")
print(f"  Scenarios included: {sc_ids_mm}")
print(f"  S4 excluded (solver artifact)")

solver_mm = pywraplp.Solver.CreateSolver("SCIP")
solver_mm.SetTimeLimit(300_000)

x_mm = [solver_mm.BoolVar(f"x_{j}") for j in J_idx_mm]
t_mm = solver_mm.NumVar(0, solver_mm.infinity(), "t")

y_s = {}
u_s = {}
for s_id in sc_ids_mm:
    y_s[s_id] = {(i, j): solver_mm.BoolVar(f"y_{s_id}_{i}_{j}") for (i, j) in vp_mm}
    u_s[s_id] = [solver_mm.BoolVar(f"u_{s_id}_{i}") for i in I_idx_mm]

for s_id in sc_ids_mm:
    d_vec = sc_demands[s_id]
    M_s   = float(r_mm * max(d_vec))

    # Coverage balance
    for i in I_idx_mm:
        if nb_i_mm[i]:
            solver_mm.Add(sum(y_s[s_id][i, j] for j in nb_i_mm[i]) + u_s[s_id][i] == 1)
        else:
            solver_mm.Add(u_s[s_id][i] == 1)

    # Link y to shared x
    for (i, j) in vp_mm:
        solver_mm.Add(y_s[s_id][i, j] <= x_mm[j])

    # Capacity
    for j in J_idx_mm:
        if nb_j_mm[j]:
            solver_mm.Add(sum(d_vec[i] * y_s[s_id][i, j] for i in nb_j_mm[j]) <= CAP_mm)

    # Regret constraint: t >= travel_s(x) + penalty_s(x) - best*_s
    travel_expr  = sum(float(d_vec[i]) * float(D[i, j]) * y_s[s_id][i, j] for (i, j) in vp_mm)
    penalty_expr = sum(float(M_s) * float(d_vec[i]) * u_s[s_id][i] for i in I_idx_mm)
    solver_mm.Add(t_mm >= travel_expr + penalty_expr - scenario_best_mm[s_id])

# Budget
solver_mm.Add(sum(x_mm[j] for j in J_idx_mm) == K_mm)

# Minimize worst-case regret
obj_mm = solver_mm.Objective()
obj_mm.SetCoefficient(t_mm, 1.0)
obj_mm.SetMinimization()

t0_mm     = time.time()
status_mm = solver_mm.Solve()
elapsed_mm = round(time.time() - t0_mm, 1)

if status_mm in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
    xmm_sites   = frozenset(j for j in J_idx_mm if x_mm[j].solution_value() > 0.5)
    xmm_site_ids = set(sites["site_id"].iloc[j] for j in xmm_sites)
    print(f"  Status: {'Optimal' if status_mm == pywraplp.Solver.OPTIMAL else 'Feasible'}  "
          f"t={round(t_mm.solution_value(), 2)}  time={elapsed_mm}s")
else:
    print(f"  INFEASIBLE/ERROR  time={elapsed_mm}s")
    xmm_sites    = frozenset()
    xmm_site_ids = set()

# ── Evaluate x*_minimax under each scenario ──────────────────────────────────
print("\n── Evaluating x*_minimax under all scenarios ──────────────────")
xmm_costs = {}
for s_id in sc_ids_mm:
    ev_mm = evaluate_fixed(xmm_sites, sc_demands[s_id])
    xmm_costs[s_id] = ev_mm["travel_cost"]
    print(f"  {s_id}: x*_MM travel = {round(xmm_costs[s_id], 2)}")

# ── Comparison table ─────────────────────────────────────────────────────────
print("\n── Regret Comparison: x*_A vs x*_Minimax ──────────────────────")
print(f"  {'Scenario':<35} {'Best*':>8} {'xA cost':>10} {'xA reg%':>8} {'xMM cost':>10} {'xMM reg%':>9}")
print(f"  {'-'*80}")

rows_mm = []
for s_id in sc_ids_mm:
    best     = scenario_best_mm[s_id]
    cost_A   = scenario_xA_cost[s_id]
    cost_MM  = xmm_costs[s_id]
    reg_A    = cost_A  - best
    reg_MM   = cost_MM - best
    pct_A    = round(reg_A  / best * 100, 1) if best > 0 else 0
    pct_MM   = round(reg_MM / best * 100, 1) if best > 0 else 0
    label    = scenarios[s_id]["label"]
    print(f"  {label:<35} {best:>8,.0f} {cost_A:>10,.0f} {pct_A:>7.1f}% {cost_MM:>10,.0f} {pct_MM:>8.1f}%")
    rows_mm.append({
        "Scenario":        label,
        "Best* cost":      round(best, 2),
        "x*_A cost":       round(cost_A, 2),
        "x*_A regret":     round(reg_A, 2),
        "x*_A regret %":   pct_A,
        "x*_MM cost":      round(cost_MM, 2),
        "x*_MM regret":    round(reg_MM, 2),
        "x*_MM regret %":  pct_MM,
    })

df_mm = pd.DataFrame(rows_mm)

max_reg_A  = max(r["x*_A regret"]  for r in rows_mm)
max_reg_MM = max(r["x*_MM regret"] for r in rows_mm)
overlap    = len(xA_open_ids & xmm_site_ids)

print(f"\n  Max regret x*_A:       {round(max_reg_A, 2):,.2f}")
print(f"  Max regret x*_minimax: {round(max_reg_MM, 2):,.2f}")
if max_reg_A > 0:
    print(f"  Regret reduction:      {round((max_reg_A - max_reg_MM)/max_reg_A*100, 1)}%")
print(f"  Sites shared between x*_A and x*_minimax: {overlap}/20")
print(f"  x*_minimax site IDs: {sorted(xmm_site_ids)}")
print(f"────────────────────────────────────────────────────────────────")

df_mm.to_csv("minimax_regret_comparison.csv", index=False)
print("\nSaved minimax_regret_comparison.csv")

── Solving minimax-regret MILP ─────────────────────────────────
  Scenarios included: ['S1_baseline', 'S3_tenure_weighted']
  S4 excluded (solver artifact)
  Status: Feasible  t=1027627.64  time=299.2s

── Evaluating x*_minimax under all scenarios ──────────────────
  S1_baseline: x*_MM travel = 5126.56
  S3_tenure_weighted: x*_MM travel = 11604.75

── Regret Comparison: x*_A vs x*_Minimax ──────────────────────
  Scenario                               Best*    xA cost  xA reg%   xMM cost  xMM reg%
  --------------------------------------------------------------------------------
  Baseline (10% uniform stock)           2,513      2,513     0.0%      5,127    104.0%
  Tenure-weighted need (Scenario B)      5,953      9,161    53.9%     11,605     95.0%

  Max regret x*_A:       3,208.17
  Max regret x*_minimax: 5,652.13
  Regret reduction:      -76.2%
  Sites shared between x*_A and x*_minimax: 0/20
  x*_minimax site IDs: ['J0', 'J1', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', '

In [17]:
# ================================
# CELL 16: Folium maps
# ================================
import folium

# Map 1: MILP solution
m = folium.Map(location=[37.397, -122.078], zoom_start=13, tiles="OpenStreetMap")

for _, row in result.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(4, min(10, row["demand_A"] / 12)),
        color="#1D9E75", fill=True, fill_color="#5DCAA5",
        fill_opacity=0.7, weight=1.5,
        popup=folium.Popup(
            f"<b>Block group:</b> {row['block_group']}<br>"
            f"<b>EV demand (A):</b> {row['demand_A']:.1f}<br>"
            f"<b>EV demand (B):</b> {row['demand_B']:.1f}<br>"
            f"<b>Renter share:</b> {row['renter_share']:.2f}",
            max_width=220)
    ).add_to(m)

for _, row in solution_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=12, color="#185FA5", fill=True,
        fill_color="#378ADD", fill_opacity=0.9, weight=2.5,
        popup=folium.Popup(
            f"<b>Station {row['site_id']}</b><br>"
            f"<b>Assigned:</b> {row['n_assigned']}<br>"
            f"<b>Demand:</b> {row['stn_demand']}<br>"
            f"<b>Load:</b> {row['load_pct']}%",
            max_width=200)
    ).add_to(m)
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#042C53;white-space:nowrap;margin-top:-6px;margin-left:14px;">{row["site_id"]}</div>',
            icon_size=(40, 12), icon_anchor=(0, 6))
    ).add_to(m)

legend_html = """
<div style="position:fixed;bottom:30px;right:30px;z-index:1000;
     background:white;border:1px solid #ccc;border-radius:8px;
     padding:12px 16px;font-size:13px;font-family:sans-serif;line-height:2;">
  <b>Mountain View EV Charging — CFLP Solution</b><br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#378ADD;margin-right:6px;vertical-align:middle;"></span>Charging station (20 built)<br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#5DCAA5;margin-right:6px;vertical-align:middle;"></span>Demand point (block group)<br>
  <span style="font-size:11px;color:#888;">Click any point for details</span>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))
m.save("Mountain_View_CFLP_map.html")

# Map 2: Heuristic solution
m2 = folium.Map(location=[37.397, -122.078], zoom_start=13, tiles="OpenStreetMap")

for _, row in result.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(4, min(10, row["demand_A"] / 12)),
        color="#1D9E75", fill=True, fill_color="#5DCAA5",
        fill_opacity=0.7, weight=1.5,
        popup=folium.Popup(f"<b>Block group:</b> {row['block_group']}<br><b>EV demand (A):</b> {row['demand_A']:.1f}", max_width=200)
    ).add_to(m2)

for _, row in heuristic_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=12, color="#185FA5", fill=True,
        fill_color="#378ADD", fill_opacity=0.9, weight=2.5,
        popup=folium.Popup(
            f"<b>Station {row['site_id']}</b><br>"
            f"<b>Assigned:</b> {row['n_assigned']}<br>"
            f"<b>Load:</b> {row['load_pct']}%",
            max_width=200)
    ).add_to(m2)
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#042C53;white-space:nowrap;margin-top:-6px;margin-left:14px;">{row["site_id"]}</div>',
            icon_size=(40, 12), icon_anchor=(0, 6))
    ).add_to(m2)

legend_html2 = """
<div style="position:fixed;bottom:30px;right:30px;z-index:1000;
     background:white;border:1px solid #ccc;border-radius:8px;
     padding:12px 16px;font-size:13px;font-family:sans-serif;line-height:2;">
  <b>Mountain View EV Charging — Heuristic Solution</b><br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#378ADD;margin-right:6px;vertical-align:middle;"></span>Charging station (active)<br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#5DCAA5;margin-right:6px;vertical-align:middle;"></span>Demand point (block group)<br>
  <span style="font-size:11px;color:#888;">Click any point for details</span>
</div>"""
m2.get_root().html.add_child(folium.Element(legend_html2))
m2.save("Mountain_View_heuristic_map.html")

print("Saved Mountain_View_CFLP_map.html and Mountain_View_heuristic_map.html")
# from google.colab import files
# files.download("Mountain_View_CFLP_map.html")
# files.download("Mountain_View_heuristic_map.html")


Saved Mountain_View_CFLP_map.html and Mountain_View_heuristic_map.html


In [18]:
# ================================
# CELL 17: File verification and download
# ================================
import os
import zipfile
# from google.colab import files

expected_files = [
    "Mountain_View_demand_points.csv",
    "Mountain_View_candidate_sites.csv",
    "Mountain_View_existing_stations.csv",
    "Mountain_View_HCVI.csv",
    "Mountain_View_site_tiers.csv",
    "Mountain_View_CFLP_solution.csv",
    "Mountain_View_CFLP_solution_B.csv",
    "Mountain_View_heuristic_solution.csv",
    "Mountain_View_scenario_analysis.csv",
    "Mountain_View_equity_results.csv",
    "Mountain_View_road_vs_haversine.csv",
    "sensitivity_K.csv",
    "sensitivity_Q.csv",
    "sensitivity_r.csv",
    "sensitivity_adoption.csv",
    "minimax_regret_comparison.csv",
    "Mountain_View_queueing_results.csv",
    "queueing_sensitivity_m.csv",
    "Mountain_View_queueing_results_B.csv",
    "Mountain_View_reliability_outage.csv",
]

print("File status:")
for f in expected_files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) if exists else 0
    status = f"✓ ({size:,} bytes)" if exists else "✗ MISSING"
    print(f"  {status:25} {f}")

with zipfile.ZipFile("ev_cflp_data.zip", "w") as zf:
    for fname in expected_files:
        if os.path.exists(fname):
            zf.write(fname)
            print(f"  Added: {fname}")
        else:
            print(f"  MISSING: {fname}")

# files.download("ev_cflp_data.zip")
print("Done.")

File status:
  ✗ MISSING                 Mountain_View_demand_points.csv
  ✗ MISSING                 Mountain_View_candidate_sites.csv
  ✗ MISSING                 Mountain_View_existing_stations.csv
  ✗ MISSING                 Mountain_View_HCVI.csv
  ✗ MISSING                 Mountain_View_site_tiers.csv
  ✓ (1,278 bytes)           Mountain_View_CFLP_solution.csv
  ✓ (1,289 bytes)           Mountain_View_CFLP_solution_B.csv
  ✓ (1,282 bytes)           Mountain_View_heuristic_solution.csv
  ✓ (449 bytes)             Mountain_View_scenario_analysis.csv
  ✓ (273 bytes)             Mountain_View_equity_results.csv
  ✓ (3,451 bytes)           Mountain_View_road_vs_haversine.csv
  ✓ (249 bytes)             sensitivity_K.csv
  ✓ (208 bytes)             sensitivity_Q.csv
  ✓ (189 bytes)             sensitivity_r.csv
  ✓ (201 bytes)             sensitivity_adoption.csv
  ✓ (255 bytes)             minimax_regret_comparison.csv
  ✓ (1,209 bytes)           Mountain_View_queueing_results.csv
  ✓ (